# Transfer boundaries

Reproduces cross-scale detection, the long-horizon step-aware alarm, and frozen LoRA-to-full-finetuning transfer. The FFT threshold sweep is post-hoc and label-assisted, not a deployment result.


In [1]:
from pathlib import Path
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "src/reproduction/reports.py").exists())
sys.path.insert(0, str(ROOT))
import json
import pandas as pd
from IPython.display import display
from src.reproduction.reports import run_analysis, show_report

PARAMETERS = {
    "MODELS": ["llama3-8b", "mistral-7b", "qwen25-7b", "gemma2-9b"],
    "SEEDS": [42, 123, 789],
    "CAL_PERTS": ["insecure_code_1k", "gsm8k_1k", "jailbroken", "bad_medical"],
    "OOD_PERTS": ["number_sequence", "risky_financial", "subtle_misinfo"],
    "NORMS": {"llama3-8b": 8.5, "mistral-7b": 4.6875, "qwen25-7b": 66.5, "gemma2-9b": 372.0},
    "EM_THRESH": 0.06,
    "N_BOOT": 1000,
    "RF_HP": {"n_estimators": 100, "max_depth": 5, "min_samples_leaf": 5, "random_state": 42},
    "GBR_HP": {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1, "random_state": 42},
    "RIDGE_KW": {"alpha": 1.0},
}
display(pd.DataFrame([{"Parameter": key, "Value": str(value)} for key, value in PARAMETERS.items()]))


,Parameter,Value
0,MODELS,"['llama3-8b', 'mistral-7b', 'qwen25-7b', 'gemm..."
1,SEEDS,"[42, 123, 789]"
2,CAL_PERTS,"['insecure_code_1k', 'gsm8k_1k', 'jailbroken',..."
3,OOD_PERTS,"['number_sequence', 'risky_financial', 'subtle..."
4,NORMS,"{'llama3-8b': 8.5, 'mistral-7b': 4.6875, 'qwen..."
5,EM_THRESH,0.06
6,N_BOOT,1000
7,RF_HP,"{'n_estimators': 100, 'max_depth': 5, 'min_sam..."
8,GBR_HP,"{'n_estimators': 100, 'max_depth': 3, 'learnin..."
9,RIDGE_KW,{'alpha': 1.0}


## Cross-scale detection

The 14B probes store their uniform-N=1000 calibration runs under unsuffixed dataset names. Probe-specific layers and learning-rate paths are resolved by the analysis configuration.


In [2]:
report = run_analysis("cross_scale", PARAMETERS)
checks = show_report(report)
if not checks.empty:
    display(checks)
display(pd.DataFrame.from_dict(report["state"]["PROBE_CONFIGS"], orient="index"))


cross scale


,Probe,Mode,ML,FNR (%),FPR (%),Acc (%)
0,Qwen 14B,within,RF,16.4,0.0,92.3
1,Qwen 14B,cross,Ridge,34.5,0.0,83.8
2,Phi-4 14B,within,GBR,0.0,5.9,96.6
3,Phi-4 14B,cross,GBR,0.0,7.4,95.7
4,Phi-4 14B,cross,Ridge,0.0,14.7,91.5


cross scale full


,Probe,Mode,ML,FNR (%),FPR (%),FN,Acc (%)
0,Qwen 14B,within,Ridge,30.9,0.0,17,85.5
1,Qwen 14B,within,GBR,16.4,3.2,9,90.6
2,Qwen 14B,within,RF,16.4,0.0,9,92.3
3,Qwen 14B,cross,Ridge,34.5,0.0,19,83.8
4,Qwen 14B,cross,GBR,94.5,3.2,52,53.8
5,Qwen 14B,cross,RF,100.0,0.0,55,53.0
6,Phi-4 14B,within,Ridge,0.0,8.8,0,94.9
7,Phi-4 14B,within,GBR,0.0,5.9,0,96.6
8,Phi-4 14B,within,RF,0.0,5.9,0,96.6
9,Phi-4 14B,cross,Ridge,0.0,14.7,0,91.5


,Table,Matches paper artifact
0,tab_cross_scale.tex,True
1,tab_cross_scale_full.tex,True


,norm,lr_prefix,label,ood_seeds
qwen25-14b,71.91,,Qwen 14B,"[42, 123, 789]"
phi4-14b,60.75,"{'insecure_code': 'lr2e-5/', 'gsm8k': '', 'jai...",Phi-4 14B,"[42, 123, 789]"


## Long-horizon step-aware alarm

A single Bitext anchor seed enters training in each of three folds. Risky-financial 5k and Alpaca are test datasets; the alarm probability threshold is 0.5.


In [3]:
report = run_analysis("cross_regime", PARAMETERS)
checks = show_report(report)
if not checks.empty:
    display(checks)
display(pd.DataFrame(report["state"]["TEST_DATASETS"]))


cross regime


,Dataset,Features,Model,Seeds,Max prob (%),FNR (%),FPR (%),Onset
0,None,"|\text{PC1}|, step, |\text{PC1}|/\text{step}",LLaMA,3,96.5,0.0 ± 0.0,---,50--50
1,None,"|\text{PC1}|, step, |\text{PC1}|/\text{step}",Mistral,3,99.7,0.0 ± 0.0,---,20--20
2,None,"|\text{PC1}|, step, |\text{PC1}|/\text{step}",Qwen,3,98.7,0.0 ± 0.0,---,100--100
3,None,"|\text{PC1}|, step, |\text{PC1}|/\text{step}",Gemma,3,99.9,0.0 ± 0.0,---,50--50
4,None,"|\text{PC1}|, step, |\text{PC1}|/\text{step}",LLaMA,3,9.3,---,0.0 ± 0.0,---
5,None,"|\text{PC1}|, step, |\text{PC1}|/\text{step}",Mistral,3,40.4,---,0.0 ± 0.0,---
6,None,"|\text{PC1}|, step, |\text{PC1}|/\text{step}",Qwen,3,43.2,---,0.0 ± 0.0,---
7,None,"|\text{PC1}|, step, |\text{PC1}|/\text{step}",Gemma,3,55.9,---,2.8 ± 1.6,50--50


cross regime variants


,Model,scalar+step,7D+step (logistic),7D+step (RF)
0,LLaMA,0.0 ± 0.0,95.0 ± 0.0,1.7 ± 0.0
1,Mistral,0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0
2,Qwen,0.0 ± 0.0,46.7 ± 19.2,0.0 ± 0.0
3,Gemma,2.8 ± 1.6,33.3 ± 47.1,48.9 ± 40.9


,Table,Matches paper artifact
0,tab_cross_regime.tex,True
1,tab_cross_regime_variants.tex,True


,name,seeds,label,type
0,risky_financial_5k,"[42, 123, 789]",risky\_financial\_advice 5k,dangerous
1,alpaca,"[99, 123, 789]",Alpaca 5k,benign


## Frozen FFT transfer and diagnostic threshold sweep

FFT learning rates are 1e-5 except Mistral risky-financial and number-sequence runs (5e-6). The recorded FFT normalization uses 4.688 for Mistral, versus 4.6875 in the LoRA headline analysis.


In [4]:
report = run_analysis("fft_transfer", {**PARAMETERS, "NORMS": {**PARAMETERS["NORMS"], "mistral-7b": 4.688}})
checks = show_report(report)
if not checks.empty:
    display(checks)


fft error breakdown


,Detector,Model,Subtle misinfo,Risky fin.,Number seq.
0,Our 7D + RF,LLaMA,12 / 0,0 / 0,23 / 0
1,Our 7D + RF,Mistral,0 / 0,1 / 0,0 / 0
2,Our 7D + RF,Qwen,9 / 0,30 / 0,0 / 0
3,Our 7D + RF,Gemma,0 / 0,0 / 0,8 / 0
4,Our 7D + GBR,LLaMA,3 / 0,0 / 2,23 / 0
5,Our 7D + GBR,Mistral,0 / 0,0 / 0,0 / 0
6,Our 7D + GBR,Qwen,12 / 0,36 / 0,0 / 0
7,Our 7D + GBR,Gemma,0 / 0,0 / 0,8 / 0
8,Our 7D + Ridge,LLaMA,8 / 0,1 / 2,23 / 0
9,Our 7D + Ridge,Mistral,0 / 0,0 / 0,0 / 0


fft threshold sweep


,Features,ML,AUROC,Deployed (τ = 6%) / FNR,Deployed (τ = 6%) / FPR,Column 6,\multicolumn{3}{c}{Post-hoc oracle τ^{*}} / τ^{*},\multicolumn{3}{c}{Post-hoc oracle τ^{*}} / FNR,\multicolumn{3}{c}{Post-hoc oracle τ^{*}} / FPR
0,Our 7D,RF,0.931,21.7,0.0,,0.035,8.9,2.6
1,Our 7D,GBR,0.923,21.5,2.6,,0.045,11.8,3.9
2,Our 7D,Ridge,0.922,31.2,2.6,,0.025,8.1,3.9
3,|PC1|,RF,0.850,34.8,1.3,,0.027,23.3,1.3
4,|PC1|,GBR,0.863,33.8,1.3,,0.030,24.3,2.6
5,|PC1|,Ridge,0.916,48.4,2.6,,0.022,9.7,2.6


fft transfer


,Features,ML,Acc (%),FNR % [95% CI],FPR % [95% CI],FN,FP,AUROC
0,Our 7D,RF,81.9,"21.7 [11.5, 32.8]","0.0 [0.0, 0.0]",83,0,0.931
1,Our 7D,GBR,81.7,"21.5 [10.9, 34.6]","2.6 [0.0, 8.6]",82,2,0.923
2,Our 7D,Ridge,73.6,"31.2 [16.0, 47.8]","2.6 [0.0, 9.8]",119,2,0.922
3,|PC1| only,RF,70.7,"34.8 [19.9, 51.0]","1.3 [0.0, 5.9]",133,1,0.850
4,|PC1| only,GBR,71.6,"33.8 [20.4, 50.2]","1.3 [0.0, 6.1]",129,1,0.863
5,|PC1| only,Ridge,59.2,"48.4 [31.4, 65.8]","2.6 [0.0, 10.4]",185,2,0.916


fft transfer main


,Features,ML,Deployed (τ = 6%) / Acc,Deployed (τ = 6%) / FNR,Deployed (τ = 6%) / FPR,Deployed (τ = 6%) / AUROC,Column 7,\multicolumn{2}{c}{Post-hoc τ^{*}} / τ^{*},\multicolumn{2}{c}{Post-hoc τ^{*}} / FNR
0,Our 7D,RF,81.9,21.7,0.0,0.931,,0.035,8.9
1,Our 7D,GBR,81.7,21.5,2.6,0.923,,0.045,11.8
2,Our 7D,Ridge,73.6,31.2,2.6,0.922,,0.025,8.1
3,|PC1|,Ridge,59.2,48.4,2.6,0.916,,0.022,9.7


,Table,Matches paper artifact
0,tab_fft_error_breakdown.tex,True
1,tab_fft_threshold_sweep.tex,True
2,tab_fft_transfer.tex,True
3,tab_fft_transfer_main.tex,True
